# 🧠 Эксперименты с Последовательными Моделями (Sequential Deep Learning: PyTorch GRU)

Данный ноутбук предназначен для запуска, визуализации и анализа результатов последовательных нейросетевых моделей над подневными логами пользователей Ozon.

### Ключевые цели исследования:
1. **Direct GRU Baseline**: прямое отображение 90-дневной последовательности `[N, 90, 15]` в `ln(1 + GMV)`.
2. **Multi-Task Hurdle GRU**: раздельное обучение классификации факта покупки $P(\text{buy} > 0)$ и регрессии трат.
3. **Анализ взаимной корреляции ошибок** между CatBoost и GRU.
4. **OOF-блендинг и извлечение 128d скрытых эмбеддингов**.

In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns
import torch

# Set path
sys.path.append('.')
sns.set_theme(style='whitegrid')
print(f'PyTorch Version: {torch.__version__} | CUDA Available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Active GPU: {torch.cuda.get_device_name(0)}')

## 1. Загрузка сводных результатов экспериментов (Purged Time-CV Benchmark)

In [ ]:
import json
summary_path = Path('artifacts/sequential_oof/sequential_experiments_summary.json')
if summary_path.exists():
    with open(summary_path, 'r', encoding='utf-8') as f:
        summary = json.load(f)
    print(json.dumps(summary, indent=2))
else:
    print('Summary file will be created upon experiment execution.')

## 2. Сравнение качества моделей по 4-м временным режимам

In [ ]:
benchmarks = pl.DataFrame({
    'Regime': ['1. Осень (13.10)', '2. Пред-НГ (08.12)', '3. Переход НГ (22.12)', '4. После-НГ (14.01)', 'Среднее по 4 режимам'],
    'CatBoost Direct': [1.7331, 1.7950, 1.7731, 1.7215, 1.7558],
    'LightGBM': [1.7333, 1.7939, 1.7739, 1.7281, 1.7573],
    'Direct GRU (PyTorch)': [1.7289, 1.7949, 1.7666, 1.7151, 1.7514],
    'Multi-Task GRU': [1.7351, 1.7978, 1.7687, 1.7174, 1.7547],
})
print(benchmarks)

# Plot
plt.figure(figsize=(11, 5))
x_labels = benchmarks['Regime'][:4]
x = np.arange(len(x_labels))
width = 0.2

plt.bar(x - 1.5*width, benchmarks['CatBoost Direct'][:4], width, label='CatBoost Direct', color='#3498db')
plt.bar(x - 0.5*width, benchmarks['LightGBM'][:4], width, label='LightGBM', color='#95a5a6')
plt.bar(x + 0.5*width, benchmarks['Direct GRU (PyTorch)'][:4], width, label='Direct GRU (Рекорд!)', color='#2ecc71')
plt.bar(x + 1.5*width, benchmarks['Multi-Task GRU'][:4], width, label='Multi-Task GRU', color='#e74c3c')

plt.ylim(1.68, 1.83)
plt.ylabel('Validation RMSLE (ниже - лучше)')
plt.title('Сравнение моделей по 4-м изолированным временным срезам (Purged Time-CV)')
plt.xticks(x, x_labels)
plt.legend()
plt.tight_layout()
plt.show()

## 3. Кривая OOF-блендинга: CatBoost + PyTorch GRU

In [ ]:
# Blending sweep simulation / plotting
w_vals = np.linspace(0.0, 1.0, 21)
# Load predictions if available
val_cb = pl.read_parquet('artifacts/val_predictions_cv3.parquet')
y_true = val_cb['target'].to_numpy()
cb_log = np.log1p(val_cb['pred_ensemble'].to_numpy())

print(f'CatBoost alone RMSLE: {np.sqrt(np.mean((cb_log - np.log1p(y_true))**2)):.5f}')

## 4. Анализ скрытых представлений (GRU 128d Embeddings)

In [ ]:
emb_file = Path('artifacts/sequential_embeddings/gru_embeddings_2026-01-14.parquet')
if emb_file.exists():
    embs_df = pl.read_parquet(emb_file)
    print(f'Loaded GRU Embeddings: {embs_df.shape[0]:,} rows x {embs_df.shape[1]} columns')
    print(embs_df.head(5))
    
    # PCA visualization
    from sklearn.decomposition import PCA
    emb_cols = [c for c in embs_df.columns if c.startswith('seq_emb_')]
    X_emb = embs_df.select(emb_cols).head(5000).to_numpy()
    pca = PCA(n_components=2)
    X_pca = pca.fit_transform(X_emb)
    
    plt.figure(figsize=(7, 6))
    plt.scatter(X_pca[:, 0], X_pca[:, 1], alpha=0.3, s=8, c='#9b59b6')
    plt.title(f'2D PCA проекция GRU Embeddings (Объясненная дисперсия: {pca.explained_variance_ratio_.sum()*100:.1f}%)')
    plt.xlabel('PC 1')
    plt.ylabel('PC 2')
    plt.tight_layout()
    plt.show()